# Part A — Transfer Learning on Food101

**Assignment 2 · Computer Vision**

This notebook implements **transfer learning** using three pre-trained CNN architectures on the [Food101 dataset](http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz):

| Model | Year | Key Innovation |
|---|---|---|
| GoogLeNet (Inception v1) | 2014 | Inception modules — parallel conv branches |
| MobileNet V3 | 2019 | Depthwise separable convs + SE blocks — mobile-friendly |
| ResNet-50 | 2015 | Residual skip connections — solves vanishing gradient |

**Strategy for each model:**
1. Load ImageNet pre-trained weights
2. Freeze all pre-trained layers
3. Replace the classification head with 3 fully-connected layers
4. Train only the new head on Food101
5. Evaluate and compare performance

---

## 0 · Imports & Configuration

In [1]:
import os
import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision import datasets, models, transforms

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
# Try to mount Google Drive (works in Google Colab); fall back to local path
_GDRIVE_BASE = Path("/content/drive/MyDrive/UTS/Semestre 2/Deep Learning/Assignment2")

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    if _GDRIVE_BASE.exists():
        BASE_DIR = _GDRIVE_BASE
        print(f"Google Drive mounted — using: {BASE_DIR}")
    else:
        BASE_DIR = Path(".")
        print(f"Google Drive mounted but path not found, using local: {BASE_DIR}")
except Exception:
    BASE_DIR = Path(".")
    print(f"Google Drive not available — using local path: {BASE_DIR}")

DATA_DIR   = BASE_DIR / "data/food-101"
IMG_DIR    = DATA_DIR / "images"
META_DIR   = DATA_DIR / "meta"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hardware ───────────────────────────────────────────────────────────────────
DEVICE = (
    torch.device("cuda")  if torch.cuda.is_available()  else
    torch.device("mps")   if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"Using device: {DEVICE}")

# ── Hyperparameters ────────────────────────────────────────────────────────────
NUM_CLASSES  = 101
BATCH_SIZE   = 128
NUM_EPOCHS   = 120
LEARNING_RATE = 0.001
IMG_SIZE     = 224

# Fraction of training data to use (set to 1.0 for full dataset)
TRAIN_FRACTION = 0.2   # reduce to speed up experimentation
VAL_FRACTION   = 0.1   # fraction of train split used for validation

Mounted at /content/drive
Google Drive mounted — using: /content/drive/MyDrive/UTS/Semestre 2/Deep Learning
Using device: cpu


---
## 1 · Dataset Exploration

The Food101 dataset contains **101,000 images** across 101 food categories:
- **750 training images** per class (75,750 total)
- **250 test images** per class (25,250 total)
- All images resized to a maximum side length of **512 px**

In [ ]:
# ── Load class list ────────────────────────────────────────────────────────────
with open(META_DIR / "classes.txt") as f:
    classes = [line.strip() for line in f]

print(f"Number of classes  : {len(classes)}")
print(f"First 10 classes   : {classes[:10]}")
print(f"Last  10 classes   : {classes[-10:]}")

In [ ]:
# ── Load official train / test split from meta JSON ────────────────────────────
with open(META_DIR / "train.json") as f:
    train_meta = json.load(f)
with open(META_DIR / "test.json") as f:
    test_meta  = json.load(f)

train_counts = {cls: len(train_meta[cls]) for cls in classes}
test_counts  = {cls: len(test_meta[cls])  for cls in classes}

df_counts = pd.DataFrame({"train": train_counts, "test": test_counts})
print(df_counts.describe())

In [ ]:
# ── Class distribution bar chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 4))

axes[0].bar(range(len(classes)), list(train_counts.values()), color="steelblue", width=0.8)
axes[0].set_title("Training images per class", fontsize=13)
axes[0].set_xlabel("Class index")
axes[0].set_ylabel("Count")
axes[0].axhline(750, color="red", linestyle="--", label="Expected 750")
axes[0].legend()

axes[1].bar(range(len(classes)), list(test_counts.values()), color="darkorange", width=0.8)
axes[1].set_title("Test images per class", fontsize=13)
axes[1].set_xlabel("Class index")
axes[1].set_ylabel("Count")
axes[1].axhline(250, color="red", linestyle="--", label="Expected 250")
axes[1].legend()

plt.suptitle("Food101 — Class distribution", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual sample of the dataset ───────────────────────────────────────────────
sample_classes = random.sample(classes, 12)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, cls in zip(axes.flatten(), sample_classes):
    img_path = IMG_DIR / cls
    img_file = random.choice(list(img_path.glob("*.jpg")))
    img = mpimg.imread(img_file)
    ax.imshow(img)
    ax.set_title(cls.replace("_", " ").title(), fontsize=9)
    ax.axis("off")

plt.suptitle("Food101 — Random sample of 12 classes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Image size distribution (sample 500 images) ────────────────────────────────
all_imgs = list(IMG_DIR.glob("*/*.jpg"))
sample_imgs = random.sample(all_imgs, min(500, len(all_imgs)))

widths, heights = [], []
for p in sample_imgs:
    with Image.open(p) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color="steelblue",  edgecolor="white")
axes[0].set_title("Width distribution"); axes[0].set_xlabel("pixels")
axes[1].hist(heights, bins=30, color="darkorange", edgecolor="white")
axes[1].set_title("Height distribution"); axes[1].set_xlabel("pixels")
plt.suptitle("Image size distribution (500-image sample)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Width  — mean: {np.mean(widths):.0f}  min: {min(widths)}  max: {max(widths)}")
print(f"Height — mean: {np.mean(heights):.0f}  min: {min(heights)}  max: {max(heights)}")

---
## 2 · Data Splitting & DataLoaders

The Food101 dataset provides official **train** and **test** splits via `meta/train.txt` and `meta/test.txt`. We further carve a **validation set** out of the official training split using an 80/20 stratified split.

| Split | Source | Purpose |
|---|---|---|
| Train | 80 % of official train | Optimise model weights |
| Validation | 20 % of official train | Tune hyper-parameters, early stopping |
| Test | Official test split | Final unbiased evaluation |

**Data augmentation** is applied only to the training set to improve generalisation.

In [ ]:
# ── Transforms ─────────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

post_transform = [
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
]

def train_tf(pil_transforms):
    """PIL-space steps + ToTensor/Normalize."""
    return transforms.Compose(list(pil_transforms) + post_transform)

# Consolidated to two branches: baseline resize and full augmentation
TRAIN_BRANCH_ORDER = [
    # "resize_only",
    "resize_plus_augmented",
]

TRAIN_BRANCH_TRANSFORM = {
    # "resize_only": train_tf([
    #     transforms.Resize((IMG_SIZE, IMG_SIZE))
    # ]),
    "resize_plus_augmented": train_tf([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomRotation(degrees=10),
    ]),
}

# Evaluation / test pipeline: Purely deterministic resize
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Visual check: original vs the two training branches ───────────────────────
def _denormalize_tensor(img_chw: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    x = img_chw.detach().cpu() * std + mean
    return np.clip(x.permute(1, 2, 0).numpy(), 0.0, 1.0)

_N_SAMPLES  = 5
_N_BRANCHES = len(TRAIN_BRANCH_ORDER)
_N_COLS     = 1 + _N_BRANCHES  # original + 2 branches

_preview_ds  = datasets.ImageFolder(root=IMG_DIR, transform=None)
_preview_idx = random.sample(range(len(_preview_ds)), k=_N_SAMPLES)

fig, axes = plt.subplots(
    _N_SAMPLES, _N_COLS,
    figsize=(4.0 * _N_COLS, 3.5 * _N_SAMPLES),
)

for row, idx in enumerate(_preview_idx):
    pil_img, label_idx = _preview_ds[idx]
    class_name = _preview_ds.classes[label_idx]

    # Column 0 — original PIL image
    ax = axes[row, 0]
    ax.imshow(pil_img)
    ax.axis("off")
    if row == 0:
        ax.set_title("Original", fontsize=10, fontweight="bold", pad=10)

    # Columns 1-2 — Clean resize vs Combined augmentations
    for col, branch_name in enumerate(TRAIN_BRANCH_ORDER, start=1):
        aug_tensor = TRAIN_BRANCH_TRANSFORM[branch_name](pil_img)
        ax = axes[row, col]
        ax.imshow(_denormalize_tensor(aug_tensor))
        ax.axis("off")
        if row == 0:
            ax.set_title(branch_name.replace("_", " ").title(), fontsize=10, fontweight="bold", pad=10)

plt.suptitle(
    "Data Preview: Original vs. Baseline Resize vs. Full Augmentation Branch",
    y=1.02, fontsize=14,
)
plt.tight_layout()
plt.show()


In [ ]:
# ── Full training set = concat of one ImageFolder per transform branch ─────────
train_branch_datasets = [
    datasets.ImageFolder(root=IMG_DIR, transform=TRAIN_BRANCH_TRANSFORM[name])
    for name in TRAIN_BRANCH_ORDER
]
NUM_TRAIN_BRANCHES = len(train_branch_datasets)
full_train_combined = ConcatDataset(train_branch_datasets)

# Same on-disk ordering for every branch — use for path lookups and labels
full_train_dataset_aug = train_branch_datasets[0]

full_eval_dataset = datasets.ImageFolder(root=IMG_DIR, transform=eval_transform)

class_to_idx = full_train_dataset_aug.class_to_idx
n_images_on_disk = len(full_train_dataset_aug)

print(f"Images on disk       : {n_images_on_disk}")
print(f"Training branches    : {NUM_TRAIN_BRANCHES}")
print(f"  → {', '.join(TRAIN_BRANCH_ORDER)}")
print(f"Concat train length   : {len(full_train_combined)} (= {NUM_TRAIN_BRANCHES} × on-disk)")
print(f"Classes detected     : {len(class_to_idx)}")


In [ ]:
# ── Build index sets from official meta split files ────────────────────────────
def load_split_indices(split_file, class_to_idx, img_dir):
    """Return list of dataset indices for a given meta split file."""
    with open(split_file) as f:
        paths = [line.strip() for line in f]

    # Build a path→index lookup from the dataset samples list
    path_to_idx = {
        str(Path(img_dir) / (sample[0].split(str(img_dir) + os.sep)[-1])): i
        for i, sample in enumerate(full_train_dataset_aug.samples)
    }
    # Alternative: direct reconstruction
    indices = []
    sample_paths = [s[0] for s in full_train_dataset_aug.samples]
    path_to_ds_idx = {p: i for i, p in enumerate(sample_paths)}

    for rel_path in paths:
        abs_path = str(img_dir / (rel_path + ".jpg"))
        if abs_path in path_to_ds_idx:
            indices.append(path_to_ds_idx[abs_path])
    return indices

train_indices = load_split_indices(META_DIR / "train.txt", class_to_idx, IMG_DIR)
test_indices  = load_split_indices(META_DIR / "test.txt",  class_to_idx, IMG_DIR)

print(f"Official train indices : {len(train_indices)}")
print(f"Official test  indices : {len(test_indices)}")

In [ ]:
# ── Stratified train / val split ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split

train_labels = [full_train_dataset_aug.targets[i] for i in train_indices]

tr_idx, val_idx = train_test_split(
    train_indices,
    test_size=VAL_FRACTION,
    stratify=train_labels,
    random_state=SEED,
)

print(f"Train files   : {len(tr_idx)}")
print(f"Train samples/epoch (all branches): {NUM_TRAIN_BRANCHES * len(tr_idx)}")
print(f"Val   samples : {len(val_idx)}")
print(f"Test  samples : {len(test_indices)}")


In [ ]:
# ── DataLoaders ────────────────────────────────────────────────────────────────
NUM_WORKERS = min(10, os.cpu_count() or 1)

tr_idx_loader = []
for b in range(NUM_TRAIN_BRANCHES):
    offset = b * n_images_on_disk
    tr_idx_loader.extend(offset + i for i in tr_idx)

train_loader = DataLoader(
    Subset(full_train_combined, tr_idx_loader),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    Subset(full_eval_dataset, val_idx),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    Subset(full_eval_dataset, test_indices),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")


In [ ]:
# ── Verify split class balance ─────────────────────────────────────────────────
def label_counts(indices, targets):
    return Counter(targets[i] for i in indices)

tr_counts  = label_counts(tr_idx,       full_train_dataset_aug.targets)
val_counts = label_counts(val_idx,      full_train_dataset_aug.targets)
te_counts  = label_counts(test_indices, full_train_dataset_aug.targets)

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
for ax, counts, title, colour in zip(
    axes,
    [tr_counts, val_counts, te_counts],
    ["Train", "Validation", "Test"],
    ["steelblue", "mediumseagreen", "darkorange"],
):
    ax.bar(range(NUM_CLASSES), [counts[i] for i in range(NUM_CLASSES)], color=colour, width=0.8)
    ax.set_title(f"{title} ({sum(counts.values())} images)", fontsize=11)
    ax.set_xlabel("Class index")
    ax.set_ylabel("Count")

plt.suptitle("Class balance across splits", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 3 · Model Architecture

### Custom Classification Head

For each model the original classification head is replaced with a **three-layer MLP** following the assignment specification:

```
backbone_out_features → 512 → 256 → 101
     (frozen)           FC     FC    FC (output)
                        ReLU   ReLU
                        Dropout Dropout
```

All backbone parameters are **frozen** (`requires_grad = False`); only the new head is trained.

### Architecture Overviews

| Property | GoogLeNet | MobileNet V3 | ResNet-50 |
|---|---|---|---|
| Depth | 22 layers | 48 layers | 50 layers |
| Params (full) | ~6.8 M | ~5.4 M | ~25.6 M |
| Backbone output | 1024 | 960 | 2048 |
| Key feature | Inception modules | Inverted residuals + SE | Residual connections |

In [ ]:
# def build_custom_head(in_features: int, num_classes: int = 101) -> nn.Sequential:
#     """Three fully-connected layer head as required by the assignment."""
#     return nn.Sequential(
#         nn.Linear(in_features, 512),
#         nn.ReLU(inplace=True),
#         nn.Dropout(0.4),
#         nn.Linear(512, 256),
#         nn.ReLU(inplace=True),
#         nn.Dropout(0.4),
#         nn.Linear(256, num_classes),
#     )


def build_custom_head(in_features: int, num_classes: int = 101) -> nn.Sequential:
    """Three fully-connected layer head as required by the assignment."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(512, num_classes),
    )


def freeze_backbone(model: nn.Module) -> None:
    """Freeze all parameters; new head layers will be unfrozen separately."""
    for param in model.parameters():
        param.requires_grad = False


def count_params(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
# ── GoogLeNet ──────────────────────────────────────────────────────────────────
googlenet = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1)
freeze_backbone(googlenet)
googlenet.fc = build_custom_head(googlenet.fc.in_features)  # replace head (unfrozen by default)
googlenet.aux_logits = False  # disable auxiliary classifiers during transfer
googlenet = googlenet.to(DEVICE)

total, trainable = count_params(googlenet)
print(f"GoogLeNet  — total params: {total:,}  |  trainable: {trainable:,}")
#print(googlenet)

In [ ]:
# ── MobileNet V3 Large ─────────────────────────────────────────────────────────
mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
freeze_backbone(mobilenet)
# MobileNetV3 classifier is: [Linear(960,1280), Hardswish, Dropout, Linear(1280,1000)]
in_features_mob = mobilenet.classifier[0].in_features
mobilenet.classifier = build_custom_head(in_features_mob)
mobilenet = mobilenet.to(DEVICE)

total, trainable = count_params(mobilenet)
print(f"MobileNetV3 — total params: {total:,}  |  trainable: {trainable:,}")
#print(mobilenet)

In [ ]:
# ── ResNet-50 ──────────────────────────────────────────────────────────────────
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
freeze_backbone(resnet)
resnet.fc = build_custom_head(resnet.fc.in_features)
resnet = resnet.to(DEVICE)

total, trainable = count_params(resnet)
print(f"ResNet-50  — total params: {total:,}  |  trainable: {trainable:,}")
#print(resnet)

---
## 4 · Training

Each model is trained independently using:
- **Loss function**: Cross-Entropy (standard for multi-class classification)
- **Optimiser**: Adam with `lr=1e-3`
- **LR scheduler**: ReduceLROnPlateau — halves the LR when validation loss stagnates
- **Metrics tracked per epoch**: training loss, validation loss, training accuracy, validation accuracy

In [ ]:
def train_one_epoch(model, loader, criterion, optimiser):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, leave=False, desc="  train"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimiser.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, leave=False, desc="  eval "):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


def train_model(model, model_name, train_loader, val_loader, epochs=NUM_EPOCHS):
    criterion  = nn.CrossEntropyLoss()
    optimiser  = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode="min", factor=0.5, patience=2)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0

    print(f"\n{'='*55}")
    print(f"  Training: {model_name}")
    print(f"{'='*55}")

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc   = train_one_epoch(model, train_loader, criterion, optimiser)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_path = CHECKPOINT_DIR / f"{model_name}_best.pth"
            torch.save(model.state_dict(), best_path)
            print(f"    → new best val_acc={val_acc:.4f}  saved {best_path}")

        last_ckpt = {
            "epoch": epoch,
            "model_name": model_name,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimiser.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "history": {k: list(v) for k, v in history.items()},
        }
        torch.save(last_ckpt, CHECKPOINT_DIR / f"{model_name}_checkpoint_last.pt")

        print(
            f"Epoch {epoch:>2}/{epochs}  "
            f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
            f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
            f"[{elapsed:.0f}s]"
        )

    print(f"\nBest val accuracy: {best_val_acc:.4f}")
    print(f"Best weights: {CHECKPOINT_DIR / f'{model_name}_best.pth'}")
    return history

### Training: GoogLeNet (Inception v1)

Classic 2014 architecture with **Inception modules** (multi-scale parallel convolutions). Here the backbone is frozen and only the custom 3-layer head learns Food101-specific features. Expect moderate parameter count and solid baseline accuracy.

In [ ]:
history_googlenet = train_model(googlenet,  "googlenet",  train_loader, val_loader)

  eval :   0%|          | 0/60 [00:00<?, ?it/s]

    → new best val_acc=0.4275  saved checkpoints/googlenet_best.pth
Epoch  4/120  train_loss=2.6878  train_acc=0.3428  val_loss=2.3085  val_acc=0.4275  [267s]


  train:   0%|          | 0/533 [00:00<?, ?it/s]

  eval :   0%|          | 0/60 [00:00<?, ?it/s]

Epoch  5/120  train_loss=2.6534  train_acc=0.3500  val_loss=2.3256  val_acc=0.4246  [261s]


...

Epoch 10/120  train_loss=2.5789  train_acc=0.3662  val_loss=2.2420  val_acc=0.4412  [264s]


  train:   0%|          | 0/533 [00:00<?, ?it/s]

  eval :   0%|          | 0/60 [00:00<?, ?it/s]

    → new best val_acc=0.4451  saved checkpoints/googlenet_best.pth
Epoch 11/120  train_loss=2.5664  train_acc=0.3670  val_loss=2.2354  val_acc=0.4451  [262s]


  train:   0%|          | 0/533 [00:00<?, ?it/s]

### Training: MobileNet V3 Large

**Depthwise separable convolutions** and inverted residuals keep this model lightweight while the large variant stays expressive enough for fine-grained food classes. Training updates only the new head; the backbone supplies mobile-optimised ImageNet features.

In [ ]:
history_mobilenet = train_model(mobilenet, "mobilenet", train_loader, val_loader)

### Training: ResNet-50

**Residual skip connections** ease optimisation in deep networks. ResNet-50 provides a 2048-D feature map before the head; it often leads the pack in frozen-backbone transfer on challenging classification tasks.

In [ ]:
history_resnet = train_model(resnet, "resnet", train_loader, val_loader)

---
## 5 · Training Curves — Loss & Accuracy

The plots below show **training and validation loss/accuracy over each epoch** for all three models, enabling side-by-side comparison.

In [ ]:
def plot_training_curves(histories: dict, metric: str, ylabel: str, title: str):
    """Plot train & val curves for a given metric across multiple models."""
    fig, axes = plt.subplots(1, len(histories), figsize=(6 * len(histories), 4), sharey=True)
    if len(histories) == 1:
        axes = [axes]

    colours = {"train": "steelblue", "val": "darkorange"}
    epochs = range(1, NUM_EPOCHS + 1)

    for ax, (name, hist) in zip(axes, histories.items()):
        ax.plot(epochs, hist[f"train_{metric}"], label="Train", color=colours["train"], marker="o", ms=4)
        ax.plot(epochs, hist[f"val_{metric}"],   label="Val",   color=colours["val"],   marker="s", ms=4)
        ax.set_title(name, fontsize=12)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)

    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


all_histories = {
    "GoogLeNet":   history_googlenet,
    "MobileNetV3": history_mobilenet,
    "ResNet-50":   history_resnet,
}

plot_training_curves(all_histories, metric="loss", ylabel="Cross-Entropy Loss",
                     title="Training vs Validation Loss per Epoch")

plot_training_curves(all_histories, metric="acc",  ylabel="Accuracy",
                     title="Training vs Validation Accuracy per Epoch")

In [ ]:
# ── Overlay comparison: val accuracy across all models ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, NUM_EPOCHS + 1)
palette = {"GoogLeNet": "royalblue", "MobileNetV3": "mediumseagreen", "ResNet-50": "tomato"}

for name, hist in all_histories.items():
    axes[0].plot(epochs, hist["val_loss"], label=name, color=palette[name], marker="o", ms=4)
    axes[1].plot(epochs, hist["val_acc"],  label=name, color=palette[name], marker="o", ms=4)

for ax, title, ylabel in zip(
    axes,
    ["Validation Loss Comparison", "Validation Accuracy Comparison"],
    ["Cross-Entropy Loss", "Accuracy"],
):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("All Models — Validation Metrics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6 · Test Set Evaluation

Each model is loaded with its **best checkpoint** from the `checkpoints/` folder (`{model}_best.pth`, highest validation accuracy) and evaluated on the held-out test set.

In [ ]:
criterion = nn.CrossEntropyLoss()

results = {}
for name, model, hist in [
    ("GoogLeNet",   googlenet,  history_googlenet),
    ("MobileNetV3", mobilenet,  history_mobilenet),
    ("ResNet-50",   resnet,     history_resnet),
]:
    # Load best weights from checkpoints folder
    key = name.lower().replace("-", "").replace(" ", "")
    checkpoint_path = CHECKPOINT_DIR / f"{key}_best.pth"
    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))

    test_loss, test_acc = evaluate(model, test_loader, criterion)
    results[name] = {
        "test_loss": round(test_loss, 4),
        "test_acc":  round(test_acc,  4),
        "best_val_acc": round(max(hist["val_acc"]), 4),
    }
    print(f"{name:<15} — test_loss: {test_loss:.4f}  test_acc: {test_acc:.4f}")

results_df = pd.DataFrame(results).T
print("\n", results_df)

In [ ]:
# ── Bar chart: test accuracy comparison ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
model_names = list(results.keys())
test_accs   = [results[m]["test_acc"] for m in model_names]
colours     = ["royalblue", "mediumseagreen", "tomato"]

bars = ax.bar(model_names, test_accs, color=colours, edgecolor="white", linewidth=1.2)
ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Test Accuracy — Transfer Learning Comparison", fontsize=14, fontweight="bold")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 7 · Summary & Conclusions

The table below summarises the results of transfer learning across all three architectures:


In [ ]:
summary_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
summary_df["test_acc_%"]     = (summary_df["test_acc"]     * 100).round(2)
summary_df["best_val_acc_%"] = (summary_df["best_val_acc"] * 100).round(2)
display(summary_df[["Model", "best_val_acc_%", "test_acc_%", "test_loss"]])

best_model = summary_df.loc[summary_df["test_acc"].idxmax(), "Model"]
print(f"\nBest model for Part B fine-tuning: {best_model}")

### Key Observations

- **GoogLeNet** uses Inception modules with parallel convolution branches (1×1, 3×3, 5×5) enabling efficient multi-scale feature extraction at relatively low parameter cost.
- **MobileNetV3** is the most parameter-efficient model, using depthwise separable convolutions and Squeeze-and-Excitation blocks. Despite fewer parameters, it often achieves competitive accuracy, making it suitable for mobile/edge deployment.
- **ResNet-50** benefits from residual (skip) connections that enable very deep networks to train effectively. With 2048-dimensional features from its backbone, it typically achieves the strongest transfer learning performance.
- Only the **custom 3-layer head** was trained in this experiment — the frozen backbone already provides powerful general-purpose visual features from ImageNet pretraining.
- The best-performing model above will be selected for **Part B fine-tuning**, where selected backbone layers will be progressively unfrozen.

---
*End of Part A*